# E-Commerce Decision Intelligence
## Growth, Customer Value, Product Performance & Operational Risk
**Python · SQL · Power BI · Business Analytics · Customer Analytics · Machine Learning**

### 01 — Data reliability and dimensional modelling

**Management question:** How can a marketplace grow merchandise value while improving customer value and post-purchase experience?

This project rebuilds the original Olist notebook into six decision layers: growth, customer, commercial, operations, prediction and BI. Only public Olist data is used. No confidential company figures or findings are incorporated.

**Workflow:** Raw data → quality → dimensional model → Python KPIs → customer intelligence → statistical analysis → ML → SQL → Power BI design → recommendations.

**Reproduction:** Install `requirements.txt`; put the eight files listed in `data/README.md` in `data/raw/`; run these notebooks in numerical order from the project or notebook directory. Alternatively run `python run_pipeline.py`. Seed: 42. Currency: historical BRL. Commercial population: delivered orders with item records and a purchase timestamp. GMV excludes freight and is not company accounting revenue. Read `docs/metric_dictionary.md` for every denominator.

**Contents:** 1. source audit; 2. original join defect; 3. validated model; 4. missing records and chronology; 5. reconciliation; 6. BI relationships.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_columns', 18)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
def table(name): return pd.read_csv(ROOT / 'output' / (name + '.csv'))
def chart(name): display(Image(filename=str(ROOT / 'images' / name), width=950))
def decision(evidence, interpretation, action, kpi):
    display(Markdown(f'**What the data shows:** {evidence}\n\n**Business interpretation:** {interpretation}\n\n**Recommended action:** {action}\n\n**KPI to monitor:** {kpi}'))

## 1. Audit the source before calculating business results
The input includes orders, customer records, items, payments, products, sellers, reviews and category translations. Persistent customer identity is `customer_unique_id`. Expected grains and composite keys are explicitly declared in the processing module. Missing descriptive fields are reported; broken required keys fail the build.

In [2]:
from src.pipeline import quality
model = quality(ROOT)
display(table('data_quality_audit'))
display(table('source_checksums')[['file','bytes','sha256']])

,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct,expected_key,key_unique,null_key_rows
0,orders,99441,8,0,4908,0.6169,order_id,True,0
1,customers,99441,5,0,0,0.0000,customer_id,True,0
2,items,112650,7,0,0,0.0000,order_id+order_item_id,True,0
3,payments,103886,5,0,0,0.0000,order_id+payment_sequential,True,0
4,products,32951,9,0,2448,0.8255,product_id,True,0
5,sellers,3095,4,0,0,0.0000,seller_id,True,0
6,reviews,99224,7,0,145903,21.0063,review_id+order_id,True,0
7,translation,71,2,0,0,0.0000,product_category_name,True,0


,file,bytes,sha256
0,olist_orders_dataset.csv,17654914,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c9...
1,olist_customers_dataset.csv,9033957,983a422239e1712ded753b3bf9ecf47dc73f144d306029...
2,olist_order_items_dataset.csv,15438671,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75f...
3,olist_order_payments_dataset.csv,5777138,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa9...
4,olist_products_dataset.csv,2379446,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b...
5,olist_sellers_dataset.csv,174703,1f643d2b950373b85735e7794b20986f528d7a000432e7...
6,olist_order_reviews_dataset.csv,14346950,81336b6e5183133b632a3d9ed899428e33430987fa5409...
7,product_category_name_translation.csv,2542,9da093e114e517534d7a6903b253b0d24a582830b99ea9...


## 2. Quantify the original many-to-many join defect
Joining items and payment records on order ID before aggregation multiplies rows. The comparison below repeats that defect only to measure its effect; no final metric uses the expanded table.

In [3]:
impact=table('original_join_impact')
display(impact)
r=impact.iloc[0]
decision(f"The original join overstates total payment value by BRL {r.overstatement:,.2f} ({r.overstatement_pct:.2%}).",
         'A plausible-looking chart can still have the wrong transaction grain.',
         'Use separately aggregated items and payments for all order-level reporting.',
         'Source-to-model reconciliation failures; target zero.')

,measure,correct,original_join,overstatement,overstatement_pct
0,Payment value,"16,008,872.1200","20,470,726.6600","4,461,854.5400",0.2787
1,Merchandise value,"13,591,643.7000","14,209,250.3100","617,606.6100",0.0454


**What the data shows:** The original join overstates total payment value by BRL 4,461,854.54 (27.87%).

**Business interpretation:** A plausible-looking chart can still have the wrong transaction grain.

**Recommended action:** Use separately aggregated items and payments for all order-level reporting.

**KPI to monitor:** Source-to-model reconciliation failures; target zero.

## 3. Build order and item facts with stable dimensions
The implementation validates joins (`many_to_one` or `one_to_one`) and preserves every source order. The latest review is selected deterministically for descriptive reporting, while all review rows remain auditable. Financial missing values are not silently replaced with observed zero.

In [4]:
display(table('model_inventory')[['table','rows','columns']])
display(model['FactOrders'][['order_id','number_of_items','merchandise_value','freight_value','payment_value','payment_difference']].head(10))
assert model['FactOrders'].order_id.is_unique
assert model['DimCustomer'].customer_unique_id.is_unique
assert model['DimProduct'].product_id.is_unique

,table,rows,columns
0,FactOrders,99441,41
1,FactOrderItems,112650,7
2,DimCustomer,96096,5
3,DimProduct,32951,11
4,DimSeller,3095,4
5,DimDate,774,10
6,FactPayments,103886,5
7,FactReviews,99224,7


,order_id,number_of_items,merchandise_value,freight_value,payment_value,payment_difference
0,e481f51cbdc54678b7cc49136f2d6af7,1.0000,29.9900,8.7200,38.7100,0.0000
1,53cdb2fc8bc7dce0b6741e2150273451,1.0000,118.7000,22.7600,141.4600,0.0000
2,47770eb9100c2d0c44946d9cf07ec65d,1.0000,159.9000,19.2200,179.1200,0.0000
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0000,45.0000,27.2000,72.2000,0.0000
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0000,19.9000,8.7200,28.6200,0.0000
5,a4591c265e18cb1dcee52889e2d8acc3,1.0000,147.9000,27.3600,175.2600,-0.0000
6,136cce7faa42fdb2cefd53fdc79a6098,1.0000,49.9000,16.0500,65.9500,0.0000
7,6514b8ad8028c9f2cc2374ded245783f,1.0000,59.9900,15.1700,75.1600,-0.0000
8,76c6e866289321a7c93b82b54852dc33,1.0000,19.9000,16.0500,35.9500,0.0000
9,e69bfb5eb88e0ed6a785585b27e16dbf,1.0000,149.9900,19.7700,169.7600,-0.0000


## 4. Missing records and invalid chronology
Delivery is eligible only with usable purchase, estimated and actual dates and no detected sequence defect. A missing or invalid outcome stays null; it is not classified as on time. Reviews and commerce have their own inclusion flags.

In [5]:
display(table('coverage_audit'))
display(table('date_audit'))
decision('The coverage tables expose orders without items/payments/reviews and date exceptions.',
         'Different business questions require different eligible populations.',
         'Investigate data exceptions separately and show denominator coverage beside rates.',
         'Missing-data coverage, chronology exceptions and population counts.')

,relationship,child_rows,unmatched_rows
0,orders.customer_id -> customers.customer_id,99441,0
1,items.order_id -> orders.order_id,112650,0
2,items.product_id -> products.product_id,112650,0
3,items.seller_id -> sellers.seller_id,112650,0
4,payments.order_id -> orders.order_id,103886,0
5,reviews.order_id -> orders.order_id,99224,0
6,orders without items,99441,775
7,orders without payments,99441,1
8,orders without valid review,99441,768
9,invalid chronology,99441,189


,dataset,field,missing,unparseable
0,orders,order_purchase_timestamp,0,0
1,orders,order_approved_at,160,0
2,orders,order_delivered_carrier_date,1783,0
3,orders,order_delivered_customer_date,2965,0
4,orders,order_estimated_delivery_date,0,0
5,items,shipping_limit_date,0,0
6,reviews,review_creation_date,0,0
7,reviews,review_answer_timestamp,0,0


**What the data shows:** The coverage tables expose orders without items/payments/reviews and date exceptions.

**Business interpretation:** Different business questions require different eligible populations.

**Recommended action:** Investigate data exceptions separately and show denominator coverage beside rates.

**KPI to monitor:** Missing-data coverage, chronology exceptions and population counts.

## 5. Reconcile before interpretation
All-status source totals must match the rebuilt model. Commercial GMV is subsequently restricted to delivered orders, so it is deliberately smaller than all-status merchandise. Payment differences can reflect missing item rows, vouchers/payment recording and source anomalies; the residual table does not prove a specific cause.

In [6]:
display(table('reconciliation'))
assert table('reconciliation').passed.all()
f=model['FactOrders']
display(f.loc[f.payment_difference.abs().gt(.01),['order_id','order_status','merchandise_value','freight_value','payment_value','payment_difference']].head(10))

,check,source,model,passed
0,unique orders,"99,441.0000","99,441.0000",True
1,item merchandise,"13,591,643.7000","13,591,643.7000",True
2,item freight,"2,251,909.5400","2,251,909.5400",True
3,payments,"16,008,872.1200","16,008,872.1200",True
4,delivered,"96,478.0000","96,478.0000",True
5,canceled,625.0000,625.0000,True
6,unavailable,609.0000,609.0000,True


,order_id,order_status,merchandise_value,freight_value,payment_value,payment_difference
464,8adafb3466daa5395694d3a906ff9d40,delivered,168.0000,50.0200,218.0000,-0.0200
867,bb2e64c3040ceb9b7ca2bfc602adca08,delivered,248.0000,9.3600,257.3400,-0.0200
1080,84d6d9710c8af32b5e88f2d1c14ab871,delivered,49.9000,7.7800,61.7000,4.0200
1126,74016effecaa79d592487f6a4ee47d4b,delivered,36.9900,14.5200,56.9600,5.4500
1669,239f380355f65dcb68551f07d16fc4a8,delivered,149.0000,73.6300,251.6300,29.0000
1729,4c57f545143e8865ca2347d8cba154a7,delivered,124.9900,14.6200,151.0100,11.4000
1986,6e57e23ecac1ae881286657694444267,delivered,330.0000,20.4100,333.9100,-16.5000
2277,b38b3526b8b8fdc807e8a0a42ab78573,delivered,22.3200,7.7400,30.1900,0.1300
2357,051fcda88d997d3ff86012da2a556342,delivered,49.0000,7.6000,51.7000,-4.9000
2541,8b5058499c412c6cf8d013de40e4f9d2,delivered,29.4500,19.3200,51.0200,2.2500


## 6. BI model
Use single-direction relationships: DimCustomer → FactOrders ← DimDate; FactOrders → FactOrderItems ← DimProduct/DimSeller. Do not join payments to items. Category/seller filters require explicit order-ID propagation for order-level rates. The complete relationship diagram, Power Query and DAX are in `powerbi/`.

**Next:** calculate business metrics from this validated model in notebook 02.